In [ ]:
import psycopg2
from psycopg2.extras import RealDictCursor
from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd
from typing import List, Dict, Tuple

In [ ]:
# Update with your connection details
conn = psycopg2.connect(
    host="localhost",
    database="your_db_name",
    user="your_username",
    password="your_password"
)

# Test connection
with conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM chunks;")
    count = cur.fetchone()[0]
    print(f"✓ Connected! Total chunks: {count}")

In [ ]:
import os, sys, pathlib
print("CWD:", os.getcwd())
print("Kernel python:", sys.executable)
print("Env present here?:", pathlib.Path(".env").resolve(), pathlib.Path(".env").exists())

In [ ]:
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

# Search upward from the current notebook folder and load the first `.env` it finds
env_path = find_dotenv(filename=".env", usecwd=True)
print("Loaded .env from:", env_path if env_path else "NOT FOUND")
if not env_path:
    # Fallback: compute repo root (4 levels up: notebooks -> test -> src -> repo-root)
    repo_root = Path.cwd().parents[3]
    env_path = repo_root / ".env"
    print("Fallback path:", env_path)
load_dotenv(env_path, override=False)

import os
dsn = os.getenv("SUPABASE_DB_URL")
assert dsn, "SUPABASE_DB_URL not set — check .env path/content"
print("OK: SUPABASE_DB_URL loaded")

In [ ]:
# retrieval_test_simple.ipynb
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
import os
from supabase import create_client, Client

# ---------------------------------------------------------------------
# STEP 1 — Robust .env loader (your version)
# ---------------------------------------------------------------------
env_path = find_dotenv(filename=".env", usecwd=True)
print("Loaded .env from:", env_path if env_path else "NOT FOUND")
if not env_path:
    # Fallback: compute repo root (4 levels up: notebooks -> test -> src -> repo-root)
    repo_root = Path.cwd().parents[3]
    env_path = repo_root / ".env"
    print("Fallback path:", env_path)

load_dotenv(env_path, override=False)

# ---------------------------------------------------------------------
# STEP 2 — Retrieve keys from .env
# ---------------------------------------------------------------------
SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_SERVICE_ROLE_KEY")

if not SUPABASE_URL or not SUPABASE_KEY:
    raise RuntimeError("Missing SUPABASE_URL or SUPABASE_SERVICE_ROLE_KEY in your .env file")

print("✅ SUPABASE_URL and SUPABASE_SERVICE_ROLE_KEY loaded")

# ---------------------------------------------------------------------
# STEP 3 — Create client
# ---------------------------------------------------------------------
supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

# ---------------------------------------------------------------------
# STEP 4 — Test fetch
# ---------------------------------------------------------------------
response = supabase.table("chunks").select("*").limit(5).execute()

if response.error:
    raise RuntimeError(f"❌ Supabase error: {response.error}")

data = response.data
print(f"✅ Retrieved {len(data)} rows")
for row in data:
    print("-" * 60)
    for k, v in row.items():
        print(f"{k}: {v}")